In [1]:
import sys
sys.path.append("../IMAGE-Mat/scripts/vema")
from preprocessing import preprocessing


In [2]:
import prism

In [3]:
base_dir = "../IMAGE-Mat/scripts/vema"
preprocessing_results = preprocessing(base_dir)

/home/qubix/Documents/repos/image-mat/image-materials/prism-stock-classes/../IMAGE-Mat/scripts/vema/preprocessing.py:311: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
/home/qubix/Documents/repos/image-mat/image-materials/prism-stock-classes/../IMAGE-Mat/scripts/vema/preprocessing.py:311: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
/home/qubix/Documents/repos/image-mat/image-materials/prism-stock-classes/../IMAGE-Mat/scripts/vema/preprocessing.py:311: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
/home/qubix/Documents/repos/image-mat/image-materials/prism-stock-classes/../IMAGE-Mat/scripts/vema/preprocessing.py:311: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
/home/qubix/Documents/repos/image-mat/image-

In [22]:
vehicle_nr = preprocessing_results['total_nr_vehicles_simple']
life_time_vehicles = preprocessing_results["lifetimes_vehicles"].to_xarray()

In [5]:
from survival import FoldedNormalSurvival, SurvivalMatrix

survival = FoldedNormalSurvival(life_time_vehicles)
survival_matrix = SurvivalMatrix(survival)


In [8]:
from globals.constants import _IMAGE_REGIONS
import numpy as np

In [27]:
import xarray as xr

In [56]:
def convert_vehicles(vehicle_nr):
        xr_vehicles_orig = vehicle_nr.to_xarray()
        time_series = xr_vehicles_orig.coords["time"]
        modes = np.unique([x[0] for x in xr_vehicles_orig.data_vars])
        region_dim = np.unique([x[1] for x in xr_vehicles_orig.data_vars])


        xr_vehicles = xr.DataArray(0.0, dims=("time", "mode", "region"),
                coords={"time": time_series,
                        "mode": modes,
                        "region": region_dim})
        for dv in xr_vehicles_orig.data_vars:
                xr_vehicles.loc[:, dv[0], dv[1]] = xr_vehicles_orig.data_vars[dv]
        return xr_vehicles


In [58]:
xr_vehicle_nr = convert_vehicles(vehicle_nr)

In [ ]:
    # stock_by_cohort = xr.DataArray(0.0, coords={
    #     "time": xr_vehicle_nr.coords["time"],
    #     "cohort": xr_vehicle_nr.coords["time"],
    #     "region": xr_vehicle_nrs.coords["region"],
    #     "mode": xr_vehicle_nr.coords["mode"]
    # })

In [60]:
Region = prism.NewDim("Region", coords=np.arange(1, 27))
Mode = prism.NewDim("Mode", coords=list(vehicle_nr.columns.levels[0].unique()))
Cohort = prism.NewDim("Cohort", coords=vehicle_nr.index.to_numpy())

In [61]:
def compute_historic(input_stock, stock_by_cohort, inflow, outflow_by_cohort, survival, start_simulation):
    for t in xr_vehicle_nr.coords["time"].loc[:start_simulation]:
        t_index = xr_vehicle_nr.coords["time"].index(t)
        if t_index != 0:
            prev_t = xr_vehicle_nr.coords["time"][t_index-1]
        else:
            prev_t = None
        
        stock_diff = input_stock[t] - stock_by_cohort[t].sum("Cohort")
        inflow[t] = stock_diff
        stock_by_cohort[t] = inflow[t]*survival[t, :]
        if prev_t is None:
            outflow_by_cohort[t] = stock_by_cohort[t]
        else:
            outflow_by_cohort[t] = stock_by_cohort[t]-stock_by_cohort[prev_t]

In [63]:
@prism.interface
class Stocks(prism.Model):
    start_simulation: int
    survival_matrix: SurvivalMatrix

    stock_by_cohort: prism.TimeVariable[Region, Mode, Cohort, 'count'] = prism.export(initial_value = prism.Array[Region, Mode, Cohort, 'count'](0.0)) 
    inflow: prism.TimeVariable[Region, Mode, 'count']         = prism.export(initial_value = prism.Array[Region, Mode, 'count'](0.0))   
    outflow_by_cohort: prism.TimeVariable[Region, Mode, Cohort, 'count'] = prism.export(initial_value = prism.Array[Region, Mode, Cohort, 'count'](0.0))
    input_stock: prism.TimeVariable[Region, Mode, 'count'] = ...

    def compute_initial_values(self, time: prism.Timeline):
        compute_historic(self.input_stock, self.stock_by_cohort, self.inflow,
                         self.outflow_by_cohort, self.survival, self.start_simulation)

    def compute_values(self, time: prism.Time):
        t, dt = time.t, time.dt
        stock_diff = self.input_stock[t] - self.stock_by_cohort[t].sum("Cohort")
        self.inflow[t] = stock_diff
        self.stock_by_cohort[t] = self.inflow[t]*self.survival[t, :]
        self.outflow_by_cohort[t] = self.stock_by_cohort[t]-self.stock_by_cohort[t-dt]

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()